In [0]:
%run ./_bootstrap

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from helpers.retriever_train import train_tfidf_knn, default_retriever_params
from helpers.mlflow_retriever import log_retriever_artifacts

In [0]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [0]:
# load doc_chunks
chunks_df = spark.table("workspace.med.doc_chunks")
chunks_df = chunks_df.filter(F.col("chunk_text").isNotNull() & (F.length("chunk_text") > 0))

In [0]:
# convert to pandas
chunks_pdf = chunks_df.select("chunk_id", "chunk_text").toPandas()

In [0]:
chunk_ids = chunks_pdf["chunk_id"].tolist()
texts = chunks_pdf["chunk_text"].tolist()

In [0]:
vectorizer, knn, matrix = train_tfidf_knn(texts, n_neighbors=10)

In [0]:
params = default_retriever_params(n_neighbors=10)
metadata = {"source_table": "workspace.med.doc_chunks", "num_chunks": len(chunk_ids)}

In [0]:

run_id = log_retriever_artifacts(vectorizer, knn, chunk_ids, metadata, params)
print("MLflow run_id:", run_id)